In [4]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("unsdsn/world-happiness")

print("Path to dataset files:", path)

c:\Users\özge\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\özge\.cache\kagglehub\datasets\unsdsn\world-happiness\versions\2


In [ ]:
import pandas as pd


In [3]:
df2015 = pd.read_csv("2015.csv")
df2016 = pd.read_csv("2016.csv")
df2017 = pd.read_csv("2017.csv")
df2018 = pd.read_csv("2018.csv")
df2019 = pd.read_csv("2019.csv")

In [5]:
df2015 = df2015.rename(columns={
    "Country": "country",
    "Region": "region",
    "Happiness Rank": "happiness_rank",
    "Happiness Score": "happiness_score",
    "Economy (GDP per Capita)": "gdp_per_capita",
    "Family": "social_support",
    "Health (Life Expectancy)": "healthy_life_expectancy",
    "Freedom": "freedom",
    "Trust (Government Corruption)": "perceptions_of_corruption",
    "Generosity": "generosity",
    "Dystopia Residual": "dystopia_residual",
    "Standard Error": "standard_error",
})

df2016 = df2016.rename(columns={
    "Country": "country",
    "Region": "region",
    "Happiness Rank": "happiness_rank",
    "Happiness Score": "happiness_score",
    "Economy (GDP per Capita)": "gdp_per_capita",
    "Family": "social_support",
    "Health (Life Expectancy)": "healthy_life_expectancy",
    "Freedom": "freedom",
    "Trust (Government Corruption)": "perceptions_of_corruption",
    "Generosity": "generosity",
    "Dystopia Residual": "dystopia_residual",
    "Lower Confidence Interval": "lower_ci",
    "Upper Confidence Interval": "upper_ci",
})

df2017 = df2017.rename(columns={
    "Country": "country",
    "Happiness.Rank": "happiness_rank",
    "Happiness.Score": "happiness_score",
    "Economy..GDP.per.Capita.": "gdp_per_capita",
    "Family": "social_support",
    "Health..Life.Expectancy.": "healthy_life_expectancy",
    "Freedom": "freedom",
    "Trust..Government.Corruption.": "perceptions_of_corruption",
    "Generosity": "generosity",
    "Dystopia.Residual": "dystopia_residual",
    "Whisker.high": "whisker_high",
    "Whisker.low": "whisker_low",
})

df2018 = df2018.rename(columns={
    "Country or region": "country",
    "Overall rank": "happiness_rank",
    "Score": "happiness_score",
    "GDP per capita": "gdp_per_capita",
    "Social support": "social_support",
    "Healthy life expectancy": "healthy_life_expectancy",
    "Freedom to make life choices": "freedom",
    "Perceptions of corruption": "perceptions_of_corruption",
    "Generosity": "generosity",
})

df2019 = df2019.rename(columns={
    "Country or region": "country",
    "Overall rank": "happiness_rank",
    "Score": "happiness_score",
    "GDP per capita": "gdp_per_capita",
    "Social support": "social_support",
    "Healthy life expectancy": "healthy_life_expectancy",
    "Freedom to make life choices": "freedom",
    "Perceptions of corruption": "perceptions_of_corruption",
    "Generosity": "generosity",
})

In [6]:
df2015["year"] = 2015
df2016["year"] = 2016
df2017["year"] = 2017
df2018["year"] = 2018
df2019["year"] = 2019

In [7]:
df_final = pd.concat(
    [df2015, df2016, df2017, df2018, df2019],
    ignore_index=True
)

In [8]:
core_cols = [
    "year", "country", "region", "happiness_rank", "happiness_score",
    "gdp_per_capita", "social_support", "healthy_life_expectancy",
    "freedom", "generosity", "perceptions_of_corruption", "dystopia_residual"
]

extra_cols = [c for c in df_final.columns if c not in core_cols]
df_final = df_final[core_cols + extra_cols]

In [9]:
# 2015 ve 2016'dan country -> region eşleme tablosu oluştur
region_map = (
    pd.concat([df2015[["country", "region"]], df2016[["country", "region"]]])
    .dropna()
    .drop_duplicates(subset="country")
    .set_index("country")["region"]
)

# Eksik region değerlerini bu haritadan doldur
df_final["region"] = df_final.apply(
    lambda row: region_map.get(row["country"], row["region"])
    if pd.isna(row["region"]) else row["region"],
    axis=1
)

In [10]:
bos = df_final[df_final["region"].isna()]["country"].unique()
print(f"Region bilgisi eksik ülkeler: {bos}")

Region bilgisi eksik ülkeler: <StringArray>
['Taiwan Province of China',  'Hong Kong S.A.R., China',
        'Trinidad & Tobago',          'Northern Cyprus',
          'North Macedonia',                   'Gambia']
Length: 6, dtype: str


In [11]:
manual_region = {
    "Taiwan Province of China": "Eastern Asia",
    "Hong Kong S.A.R., China": "Eastern Asia",
    "Trinidad & Tobago": "Latin America and Caribbean",
    "Northern Cyprus": "Western Europe",
    "North Macedonia": "Central and Eastern Europe",
    "Gambia": "Sub-Saharan Africa",
}

df_final["region"] = df_final.apply(
    lambda row: manual_region.get(row["country"], row["region"])
    if pd.isna(row["region"]) else row["region"],
    axis=1
)

In [12]:
bos = df_final[df_final["region"].isna()]["country"].unique()
print(f"Kalan eksik: {len(bos)} ülke → {bos}")

Kalan eksik: 0 ülke → <StringArray>
[]
Length: 0, dtype: str


Küçük bir not: 2015–2016'da bu ülkeler farklı isimle yazılmıştı — örneğin "Taiwan" → "Taiwan Province of China", "Hong Kong" → "Hong Kong S.A.R., China". O yüzden otomatik eşleşme tutmadı, manuel girmek gerekti.

In [14]:
print(f"Toplam satır: {len(df_final)}")
print(f"Sütunlar: {df_final.columns.tolist()}")
print(df_final.head())

df_final.to_csv("world_happiness_2015_2019.csv", index=False)
print("Kaydedildi!")

Toplam satır: 782
Sütunlar: ['year', 'country', 'region', 'happiness_rank', 'happiness_score', 'gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom', 'generosity', 'perceptions_of_corruption', 'dystopia_residual', 'standard_error', 'lower_ci', 'upper_ci', 'whisker_high', 'whisker_low']
   year      country          region  happiness_rank  happiness_score  \
0  2015  Switzerland  Western Europe               1            7.587   
1  2015      Iceland  Western Europe               2            7.561   
2  2015      Denmark  Western Europe               3            7.527   
3  2015       Norway  Western Europe               4            7.522   
4  2015       Canada   North America               5            7.427   

   gdp_per_capita  social_support  healthy_life_expectancy  freedom  \
0         1.39651         1.34951                  0.94143  0.66557   
1         1.30232         1.40223                  0.94784  0.62877   
2         1.32548         1.36058         

In [15]:
cd happiness_project
python birlestir.py

SyntaxError: invalid syntax (457087243.py, line 1)